# Serve an Index over MCP

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open standard that lets AI agents discover and call external tools through one uniform interface. RedisVL ships an MCP server, the `rvl mcp` command, that exposes a single existing Redis index to any MCP client through two tools:

- **`search-records`** : semantic, full-text, or hybrid retrieval over the index.
- **`upsert-records`** : add or overwrite records in the index.

The server owns the embedding model, so clients only ever send **text**. No raw vectors cross the client boundary, retrieval behavior lives entirely in one config file, and the same index can be shared with ADK, Claude Desktop, Cursor, or any other MCP client with zero custom code.

This guide walks the full loop end to end:

1. Create and load a Redis index.
2. Write the MCP config that binds the server to that index.
3. Start the RedisVL MCP server over Streamable HTTP.
4. Call `search-records` and `upsert-records` from a plain MCP client.
5. Wire the same server to a [Google ADK](https://google.github.io/adk-docs/) agent so a model can retrieve and write knowledge through MCP.

## Prerequisites

Before you begin, ensure you have:
- A running Redis instance ([Redis 8+](https://redis.io/downloads/) or [Redis Cloud](https://redis.io/cloud)) with the Search capability.
- [`uv`](https://docs.astral.sh/uv/) on your `PATH` (the server is launched with `uvx`/`rvl`).

For the complete config schema and tool contracts, see the [Run RedisVL MCP](how_to_guides/mcp.md) how-to guide.


## Install Packages

The MCP server lives behind the `mcp` extra. We also install the `sentence-transformers` extra so query and record embedding can run locally with no API key.


In [ ]:
# NBVAL_SKIP
%pip install -q "redisvl[mcp,sentence-transformers]>=0.21.0" nest_asyncio pandas

## Connect to Redis

The MCP server reads from a normal Redis URL. We use the same URL here to create the index it will serve.


In [ ]:
import os
import warnings

import nest_asyncio
try:
    import pandas as pd  # optional: only used to render result tables
except ImportError:  # keep notebook tests working without pandas installed
    pd = None

# Notebook event loops are already running; this lets the MCP client and the
# ADK runner use top-level await cleanly.
warnings.filterwarnings("ignore")
nest_asyncio.apply()

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379")

from redis import Redis

Redis.from_url(REDIS_URL).ping()
print("Connected to", REDIS_URL)

## 1. Create and Load an Index

The MCP server only ever **binds to an index that already exists**, it never creates one. So the first step is the ordinary RedisVL flow: define a schema, embed some records, and load them.

We use a tiny Redis knowledge corpus and embed each record's `text` field locally with `HFTextVectorizer` (`all-MiniLM-L6-v2`, 384 dimensions). The `doc_id` tag gives every record a stable identifier so upserts can update in place.

> **Reserved field names:** RedisVL MCP uses `id`, `score`, and a few others in its response envelope. Name your identifier field something else (here, `doc_id`) so it does not collide.


In [ ]:
from redisvl.index import SearchIndex
from redisvl.utils.vectorize import HFTextVectorizer

INDEX_NAME = "redisvl_mcp_guide"
INDEX_PREFIX = "mcp_guide"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

documents = [
    {"doc_id": "redisvl-intro", "title": "What is RedisVL",
     "text": "RedisVL is a Python client for building AI applications on Redis, "
             "with vector search, semantic caching, and semantic routing."},
    {"doc_id": "vector-search", "title": "Vector search",
     "text": "Redis stores embeddings and runs k-nearest-neighbor vector similarity "
             "search using HNSW or FLAT indexes."},
    {"doc_id": "semantic-cache", "title": "Semantic caching",
     "text": "SemanticCache stores past LLM responses and returns a cached answer when "
             "a new prompt is semantically similar to an earlier one."},
    {"doc_id": "mcp-server", "title": "RedisVL MCP server",
     "text": "The rvl mcp command serves an existing Redis index to MCP clients through "
             "the search-records and upsert-records tools."},
]

vectorizer = HFTextVectorizer(model=EMBEDDING_MODEL)

schema = {
    "index": {"name": INDEX_NAME, "prefix": INDEX_PREFIX, "storage_type": "hash"},
    "fields": [
        {"name": "doc_id", "type": "tag"},
        {"name": "title", "type": "text"},
        {"name": "text", "type": "text"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "hnsw",
                "dims": vectorizer.dims,
                "distance_metric": "cosine",
                "datatype": "float32",
            },
        },
    ],
}

index = SearchIndex.from_dict(schema, redis_url=REDIS_URL)
index.create(overwrite=True, drop=True)

records = []
for doc in documents:
    record = dict(doc)
    record["embedding"] = vectorizer.embed(doc["text"], as_buffer=True)
    records.append(record)

keys = index.load(records, id_field="doc_id")
print(f"Loaded {len(keys)} records into index '{INDEX_NAME}'")

## 2. Write the MCP Config

The config binds **one logical MCP server to one existing index**. It has three parts:

- **`server`** : how to reach Redis (`redis_url`).
- **`indexes.<id>.search`** : how the `search-records` tool queries. `type: vector` embeds the query text and runs vector similarity. (`fulltext` and `hybrid` are also supported, hybrid needs native Redis support.)
- **`indexes.<id>.runtime`** : field mappings and guardrails.
  - `vector_field_name` / `text_field_name` : which fields to search.
  - `default_embed_text_field` : the field the server embeds, both for incoming queries and for new records on upsert. This is what makes the server **embed text itself** so clients never send vectors.
  - `default_limit` / `max_limit` / `max_result_window` : cap result sizes.

We point the same `HFTextVectorizer` model at the server so its query embeddings match the vectors we stored. Because we do **not** pass `--read-only` when launching, both `search-records` and `upsert-records` are exposed.


In [ ]:
from pathlib import Path

import yaml

from redisvl.mcp import load_mcp_config

mcp_config_path = (Path.cwd() / "redisvl_mcp_guide.yaml").resolve()

mcp_config = {
    "server": {"redis_url": REDIS_URL},
    "indexes": {
        INDEX_NAME: {
            "redis_name": INDEX_NAME,
            "vectorizer": {"class": "HFTextVectorizer", "model": EMBEDDING_MODEL},
            "search": {"type": "vector"},
            "runtime": {
                "text_field_name": "text",
                "vector_field_name": "embedding",
                "default_embed_text_field": "text",
                "default_limit": 3,
                "max_limit": 10,
                "max_result_window": 100,
                # The first call loads the embedding model into memory, which can
                # take 30+ seconds. Give startup and requests plenty of room.
                "request_timeout_seconds": 120,
                "startup_timeout_seconds": 120,
            },
        }
    },
}

mcp_config_path.write_text(yaml.safe_dump(mcp_config, sort_keys=False), encoding="utf-8")

# load_mcp_config validates the file the way the server will at startup.
validated = load_mcp_config(str(mcp_config_path))
print("Wrote and validated", mcp_config_path)
print({
    "binding_id": validated.binding_id,
    "redis_name": validated.redis_name,
    "search_type": validated.binding.search.type,
})

## 3. Start the RedisVL MCP Server

The server must be running before any client can connect. We use **Streamable HTTP** transport, which works reliably in notebooks (stdio transport breaks under Jupyter/Colab because they wrap `stdout`/`stderr`).

**Option A : from a terminal (recommended for local development).** Open a separate terminal and run:

```bash
uvx --from "redisvl[mcp,sentence-transformers]" rvl mcp \
    --config redisvl_mcp_guide.yaml \
    --transport streamable-http --host 127.0.0.1 --port 8000
```

Then skip the next code cell and set `MCP_URL = "http://127.0.0.1:8000/mcp"`.

**Option B : from the notebook (required for Colab).** The next cell launches the server as a background subprocess. It first refuses to continue if the port is already taken, then waits until the server answers a real MCP handshake (confirming it is our server and the embedding model has finished loading), not merely until the socket opens.

```{warning}
Streamable HTTP and SSE are **unauthenticated by default**. Binding to a non-loopback host (`--host 0.0.0.0`) without auth now fails closed unless you pass `--allow-unauthenticated`; binding to loopback (`127.0.0.1`, as here) without auth only warns. For any networked deployment, enable JWT authentication (see Section 6 below and the [Authenticate RedisVL MCP](how_to_guides/mcp_authentication.md) guide) rather than relying on `--allow-unauthenticated`. Without `--read-only`, the `upsert-records` write tool is exposed to any client that can reach the server.
```


In [ ]:
# NBVAL_SKIP
import asyncio
import socket
import subprocess
import time

from fastmcp import Client

MCP_HOST, MCP_PORT = "127.0.0.1", 8000
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"


def port_in_use(host: str, port: int) -> bool:
    """Return True if something is already listening on host:port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


# Fail loudly on a port clash. A bare TCP probe would otherwise treat an
# unrelated server already on this port as "ready" and surface a confusing
# "Session terminated" error on the first tool call.
if port_in_use(MCP_HOST, MCP_PORT):
    raise RuntimeError(
        f"Port {MCP_PORT} is already in use. Stop whatever is bound to it, or pick a free "
        f"port by setting MCP_PORT and MCP_URL above before launching the server."
    )

# A clearer search-tool description steers the model toward real field names.
# RedisVL still appends schema-derived filter and return_fields hints.
os.environ["REDISVL_MCP_TOOL_SEARCH_DESCRIPTION"] = (
    "Search the Redis knowledge base. Fields: doc_id (tag), title (text), text (text). "
    "Only use these field names in return_fields and filters."
)

mcp_process = subprocess.Popen(
    [
        "rvl", "mcp",
        "--config", str(mcp_config_path),
        "--transport", "streamable-http",
        "--host", MCP_HOST,
        "--port", str(MCP_PORT),
    ],
    stdin=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy(),
    start_new_session=True,  # own process group, so we can stop the whole tree
)


async def wait_until_ready(url: str, timeout: float = 120.0) -> set[str]:
    """Wait for a real MCP handshake, not just an open socket.

    Connecting with an MCP client and confirming ``search-records`` is exposed
    proves the process is actually our RedisVL server (not some other listener
    on this port) and that it has finished loading the embedding model, so the
    first real call will not cold-start.
    """
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        if mcp_process.poll() is not None:
            raise RuntimeError(f"MCP server exited early (code {mcp_process.returncode})")
        try:
            async with Client(url) as client:
                tool_names = {t.name for t in await client.list_tools()}
            if "search-records" in tool_names:
                return tool_names
        except Exception as exc:  # not ready yet, keep polling
            last_error = exc
            await asyncio.sleep(1.0)
    raise RuntimeError(f"MCP server not ready after {timeout:.0f}s (last error: {last_error})")


tool_names = await wait_until_ready(MCP_URL)
print(f"MCP server ready (PID {mcp_process.pid}); tools exposed: {sorted(tool_names)}")

## 4. Call the Tools from an MCP Client

Any MCP client can now connect. We use [`fastmcp`](https://gofastmcp.com/)'s `Client` (installed with the `mcp` extra) to show exactly what crosses the wire. Notice the request is **plain text**: the server embeds it before searching.


In [ ]:
# NBVAL_SKIP
from fastmcp import Client

async with Client(MCP_URL) as client:
    tools = await client.list_tools()
    print("Tools exposed:", [t.name for t in tools])

    result = await client.call_tool(
        "search-records",
        {"query": "How do I cache LLM responses?", "limit": 3},
    )

# result.data is the structured JSON response from the server.
search_payload = result.data

# Verify the tool actually returned grounded results.
assert search_payload["results"], "search-records returned no results"
print(f"search_type={search_payload['search_type']} | {len(search_payload['results'])} results returned")

rows = [
    {**hit["record"], "score": round(hit["score"], 3), "score_type": hit["score_type"]}
    for hit in search_payload["results"]
]
pd.DataFrame(rows)

### Upsert a New Record

`upsert-records` takes a list of records, each carrying the `default_embed_text_field` (`text`) the server will embed. Passing `id_field="doc_id"` makes writes idempotent: the same `doc_id` overwrites in place rather than creating a duplicate. We send no vector, the server generates it.


In [ ]:
# NBVAL_SKIP
async with Client(MCP_URL) as client:
    upsert_result = await client.call_tool(
        "upsert-records",
        {
            "records": [
                {
                    "doc_id": "semantic-router",
                    "title": "Semantic routing",
                    "text": "SemanticRouter classifies an incoming query and routes it to the "
                            "best matching route using vector similarity over route references.",
                }
            ],
            "id_field": "doc_id",
        },
    )
    print("Upsert:", upsert_result.data)

    # The new record is immediately searchable.
    verify = await client.call_tool(
        "search-records",
        {"query": "route queries to the right topic", "limit": 2},
    )

# Verify the write landed and is retrievable.
assert upsert_result.data["keys_upserted"] == 1, upsert_result.data
assert any(
    hit["record"]["doc_id"] == "semantic-router" for hit in verify.data["results"]
), "the upserted record was not found by a follow-up search"
print("Verified: upserted", upsert_result.data["keys"], "and found it via search")

pd.DataFrame(
    {**h["record"], "score": round(h["score"], 3)} for h in verify.data["results"]
)

## 5. Use the Index from a Google ADK Agent

The same server drops straight into an agent framework. This is the pattern from the [`redisvl-mcp-rag-agent`](https://github.com/redis-applied-ai/redisvl-mcp-rag-agent) demo, reduced to its core: a [Google ADK](https://google.github.io/adk-docs/) `LlmAgent` whose only tools are the RedisVL MCP `search-records` and `upsert-records`, reached over the same Streamable HTTP endpoint.

The agent orchestrates with a chat model (here OpenAI's `gpt-4o` via [LiteLLM](https://docs.litellm.ai/)) but **retrieval and writes go through MCP**, so the model only ever sends text.

This section needs `google-adk`, `litellm`, and an `OPENAI_API_KEY`.


In [ ]:
# NBVAL_SKIP
%pip install -q "google-adk>=1.0.0" litellm

In [ ]:
# NBVAL_SKIP
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

### Build the Agent

`McpToolset` connects ADK to the running server. The `tool_filter` lists exactly which MCP tools the agent may call, here both `search-records` and `upsert-records`. The `instruction` tells the model to ground every answer in retrieved records and to write only when explicitly asked.


In [ ]:
# NBVAL_SKIP
import uuid

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.genai import types as genai_types

APP_NAME, USER_ID = "redisvl_mcp_guide", "notebook_user"

INSTRUCTION = (
    "You are a Redis knowledge assistant backed by one RedisVL index. "
    "Both your tools take plain TEXT and the server embeds it, so never construct vectors.\n"
    "- To answer, call search-records first, then ground your answer in the results and "
    "name the titles you used. If nothing relevant comes back, say so; do not invent an answer.\n"
    "- Only call upsert-records when the user clearly asks to save or correct knowledge. Put the "
    "content in the `text` field, pass id_field='doc_id' with a `doc_id`, and confirm what you wrote."
)

toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(url=MCP_URL),
    tool_filter=["search-records", "upsert-records"],
)

agent = LlmAgent(
    name="redis_mcp_agent",
    model=LiteLlm(model="openai/gpt-4o"),
    instruction=INSTRUCTION,
    tools=[toolset],
)

session_service = InMemorySessionService()
runner = Runner(app_name=APP_NAME, agent=agent, session_service=session_service)


async def ask_agent(query: str) -> dict:
    """Run one turn; return the final answer plus the MCP tool calls it made."""
    session_id = f"session-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id, state={}
    )
    message = genai_types.Content(role="user", parts=[genai_types.Part(text=query)])

    answer, tool_calls = "", []
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=message
    ):
        for call in event.get_function_calls() or []:
            tool_calls.append({"name": call.name, "args": dict(call.args or {})})
        if event.is_final_response() and event.content and event.content.parts:
            answer = "".join(part.text or "" for part in event.content.parts)
    return {"answer": answer, "tool_calls": tool_calls}


print("Agent ready, wired to", MCP_URL)

### Ask a Grounded Question

The agent calls `search-records` and answers from what Redis returns.


In [ ]:
# NBVAL_SKIP
result = await ask_agent("What does RedisVL offer for semantic caching?")
print(result["answer"])
print("\nTool calls:", result["tool_calls"])

### Ask the Agent to Save Knowledge

Because `upsert-records` is exposed, the agent can add to the corpus mid-conversation, then retrieve it.


In [ ]:
# NBVAL_SKIP
save = await ask_agent(
    "Remember this: RedisVL EmbeddingsCache stores embeddings keyed by text so you do not "
    "re-embed the same input twice. Save it with doc_id 'embeddings-cache'."
)
print(save["answer"])
print("\nTool calls:", save["tool_calls"])

check = await ask_agent("How can I avoid re-embedding the same text?")
print("\n---\n", check["answer"])

## 6. Secure the Server with JWT Authentication

Everything so far ran **unauthenticated**, which is fine for local development on `127.0.0.1`. For any networked deployment you should require a token. The HTTP transports (`streamable-http`, `sse`) support optional **JWT bearer authentication**; `stdio` is a local subprocess and is never authenticated.

In OAuth terms RedisVL MCP is a **resource server**: it validates access tokens that *your* identity provider issued, it does not run a login flow or mint tokens. On each request it checks the token **signature** (against a JWKS endpoint or a static public key), **issuer**, **audience**, and **expiry**, then gates `search-records` behind a **read scope** and `upsert-records` behind a **write scope**.

> In production you point `jwks_uri` at your IdP (Auth0, Okta, Entra, Cognito, Keycloak). To keep this notebook self-contained we mint a throwaway RSA keypair with FastMCP's `RSAKeyPair` and validate tokens against its static `public_key`, so no external IdP is required.

We start a **second** server on port 8001 with a `server.auth` block, leaving the unauthenticated server above untouched. For the full configuration reference and the gateway boundary, see the [Authenticate RedisVL MCP](how_to_guides/mcp_authentication.md) guide.

In [ ]:
# NBVAL_SKIP
import asyncio
import copy
import socket
import subprocess
import time

from fastmcp import Client
from fastmcp.client.auth import BearerAuth
from fastmcp.server.auth.providers.jwt import RSAKeyPair

# In production you point `jwks_uri` at your identity provider and validate the
# tokens it issues. Here we mint a throwaway RSA keypair locally and validate
# against its static public key, so the demo needs no external IdP.
ISSUER = "https://auth.redis.example/notebook/v2.0"
AUDIENCE = "api://redisvl-mcp"
READ_SCOPE = "kb.search.read"     # required to connect and to call search-records
WRITE_SCOPE = "kb.search.write"   # additionally required to call upsert-records

key = RSAKeyPair.generate()

# Reuse the same index and search settings, but add a server.auth block.
auth_config = copy.deepcopy(mcp_config)
auth_config["server"]["auth"] = {
    "type": "jwt",
    "public_key": key.public_key,
    "issuer": ISSUER,
    "audience": AUDIENCE,
    "required_scopes": [READ_SCOPE],   # token must carry this scope to connect
    "read_scope": READ_SCOPE,          # gate for search-records
    "write_scope": WRITE_SCOPE,        # gate for upsert-records
}

auth_config_path = (Path.cwd() / "redisvl_mcp_guide_auth.yaml").resolve()
auth_config_path.write_text(yaml.safe_dump(auth_config, sort_keys=False), encoding="utf-8")
load_mcp_config(str(auth_config_path))  # validate the auth block the way startup will

AUTH_HOST, AUTH_PORT = "127.0.0.1", 8001
AUTH_URL = f"http://{AUTH_HOST}:{AUTH_PORT}/mcp"

# Section 6 is self-contained: define the port probe here so the cell runs even
# if you used Option A in Section 3 (terminal-launched server) and skipped its cell.
def port_in_use(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


if port_in_use(AUTH_HOST, AUTH_PORT):
    raise RuntimeError(
        f"Port {AUTH_PORT} is already in use. Free it or pick another port for the "
        f"authenticated server before re-running this cell."
    )

auth_process = subprocess.Popen(
    [
        "rvl", "mcp",
        "--config", str(auth_config_path),
        "--transport", "streamable-http",
        "--host", AUTH_HOST,
        "--port", str(AUTH_PORT),
    ],
    stdin=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy(),
    start_new_session=True,
)


async def wait_until_ready_auth(url, auth, timeout=120.0):
    # Readiness probe that sends a valid token, since unauthenticated handshakes
    # are now rejected with 401.
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        if auth_process.poll() is not None:
            raise RuntimeError(
                f"Authenticated MCP server exited early (code {auth_process.returncode})"
            )
        try:
            async with Client(url, auth=auth) as client:
                tool_names = {t.name for t in await client.list_tools()}
            if "search-records" in tool_names:
                return tool_names
        except Exception as exc:  # not ready yet, keep polling
            last_error = exc
            await asyncio.sleep(1.0)
    raise RuntimeError(
        f"Authenticated MCP server not ready after {timeout:.0f}s (last error: {last_error})"
    )


connect_token = key.create_token(
    subject="notebook", issuer=ISSUER, audience=AUDIENCE, scopes=[READ_SCOPE]
)
await wait_until_ready_auth(AUTH_URL, BearerAuth(connect_token))
print(f"Authenticated MCP server ready (PID {auth_process.pid}) at {AUTH_URL}")

### Connect With and Without a Token

The same `fastmcp` client now sends a **bearer token**. We exercise three callers against the authenticated server:

- **No token** is rejected at the transport with HTTP 401.
- A **read-only token** (carries the read scope) can `search-records` but is forbidden from `upsert-records`.
- A **read+write token** (carries both scopes) can also `upsert-records`.

This is *coarse* authorization: RedisVL authenticates the caller and gates read vs write. Mapping a token's identity to a specific Redis ACL user or per-tenant query filters is a gateway concern, covered in the [Authenticate RedisVL MCP](how_to_guides/mcp_authentication.md) guide.

In [ ]:
# NBVAL_SKIP
from fastmcp.exceptions import ToolError

# 1. No token at all: rejected at the transport with HTTP 401.
try:
    async with Client(AUTH_URL) as client:
        await client.list_tools()
    raise AssertionError("expected the unauthenticated request to be rejected")
except Exception as exc:
    assert "401" in str(exc), f"expected a 401, got {exc!r}"
    print("No token         -> rejected (401)")

# 2. Read-only token: holds the read scope, so search works but upsert is forbidden.
read_token = key.create_token(
    subject="reader", issuer=ISSUER, audience=AUDIENCE, scopes=[READ_SCOPE]
)
async with Client(AUTH_URL, auth=BearerAuth(read_token)) as client:
    found = await client.call_tool("search-records", {"query": "vector search", "limit": 1})
    assert found.data["results"], "the read token should be able to search"
    print(f"Read token       -> search OK ({len(found.data['results'])} result)")
    try:
        await client.call_tool(
            "upsert-records",
            {"records": [{"doc_id": "blocked", "title": "x", "text": "blocked write"}],
             "id_field": "doc_id"},
        )
        raise AssertionError("expected the read-only token to be blocked from upsert")
    except ToolError:
        print("Read token       -> upsert FORBIDDEN (missing write scope)")

# 3. Read+write token: holds both scopes, so upsert succeeds.
rw_token = key.create_token(
    subject="editor", issuer=ISSUER, audience=AUDIENCE, scopes=[READ_SCOPE, WRITE_SCOPE]
)
async with Client(AUTH_URL, auth=BearerAuth(rw_token)) as client:
    written = await client.call_tool(
        "upsert-records",
        {"records": [{"doc_id": "auth-demo", "title": "Authenticated write",
                      "text": "This record was written through an authenticated MCP call."}],
         "id_field": "doc_id"},
    )
    assert written.data["keys_upserted"] == 1, written.data
    print(f"Read+write token -> upsert OK ({written.data['keys_upserted']} record)")

## Cleanup

Close the toolset, stop the background server (skip if you started it from a terminal, stop it there with Ctrl-C), and drop the index.


In [ ]:
import os
import signal

try:
    await toolset.close()
except NameError:
    pass  # agent section was skipped

if "mcp_process" in dir() and mcp_process.poll() is None:
    os.killpg(os.getpgid(mcp_process.pid), signal.SIGTERM)
    mcp_process.wait(timeout=10)
    print("MCP server stopped.")

if "auth_process" in dir() and auth_process.poll() is None:
    os.killpg(os.getpgid(auth_process.pid), signal.SIGTERM)
    auth_process.wait(timeout=10)
    print("Authenticated MCP server stopped.")

index.delete(drop=True)
mcp_config_path.unlink(missing_ok=True)
if "auth_config_path" in dir():
    auth_config_path.unlink(missing_ok=True)
print("Index dropped and config removed.")

## Recap

You served a single Redis index over MCP and used it several ways:

1. **Index + config** : created a RedisVL index, then bound it to the `rvl mcp` server with a small YAML config. `default_embed_text_field` makes the server embed text itself, so clients send only text.
2. **Direct client** : connected with `fastmcp` and called `search-records` and `upsert-records` over Streamable HTTP.
3. **ADK agent** : pointed a Google ADK `LlmAgent` at the same endpoint with `McpToolset`, so the model retrieves and writes knowledge through MCP with no Redis-specific code.
4. **Authenticated access** : started a second server with a `server.auth` JWT block, then watched a no-token request get rejected, a read-only token search but fail to write, and a read+write token upsert successfully.

Any MCP-compatible client (ADK, Claude Desktop, Cursor) can reuse this exact server and index. For the full config schema, tool contracts, transports, and read-only mode, see the [Run RedisVL MCP](how_to_guides/mcp.md) and [Authenticate RedisVL MCP](how_to_guides/mcp_authentication.md) how-to guides.
